# Week 1: Foundations of Generative AI
**Applied Generative AI**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

*Developed by Prachi Patel, PhD | Open Curriculum — CC BY 4.0*

---

## Learning Objectives

By the end of this notebook, you will be able to:

- **Understand** the generative AI landscape and key model families
- **Explain** how tokenization converts text to model inputs
- **Describe** the attention mechanism and its role in transformers
- **Build intuition** for how transformer-based models generate text
- **Critically assess** what LLMs can and cannot do

---
## Setup

In [ ]:
!pip install -q transformers tiktoken torch matplotlib numpy

In [ ]:
import tiktoken
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, GPT2LMHeadModel, GPT2Tokenizer

print("✅ Setup complete!")

---
## 1. The Generative AI Landscape

### A Brief History

The path to today's large language models follows a clear arc:

| Era | Milestone | Significance |
|-----|-----------|---------------|
| **Pre-2017** | RNNs, LSTMs | Sequential processing; struggled with long-range dependencies |
| **2017** | *Attention Is All You Need* | Self-attention replaces recurrence; parallelizable |
| **2018–2019** | BERT, GPT-1/2 | Pre-training at scale; encoder-decoder vs decoder-only |
| **2020** | GPT-3 | 175B parameters; few-shot learning; emergent capabilities |
| **2022** | ChatGPT | RLHF; conversational interface; broad adoption |
| **2023–Today** | Claude, Gemini, Llama, Mistral | Multimodal models; open weights; frontier competition |

The **transformer** architecture — introduced in 2017 — is the foundation. Its key innovation: **attention** allows each token to attend to every other token in the sequence, enabling models to capture long-range dependencies without the sequential bottleneck of RNNs.

### Key Model Families

| Family | Origin | Architecture | Notable Features |
|--------|--------|--------------|------------------|
| **GPT** | OpenAI | Decoder-only | Autoregressive; foundation for ChatGPT |
| **Claude** | Anthropic | Decoder-only | Constitutional AI; long context |
| **Gemini** | Google | Encoder-decoder (multimodal) | Native multimodal; reasoning |
| **Llama** | Meta | Decoder-only | Open weights; widely fine-tuned |
| **Mistral** | Mistral AI | Decoder-only | Efficient; strong open-source alternative |

Understanding these families helps you choose models for different tasks and understand the design tradeoffs (open vs. closed, encoder-decoder vs. decoder-only, context length, etc.).

---
## 2. Tokenization

LLMs operate on **tokens**, not raw characters or words. Tokenization is the process of converting text into discrete units the model can process. Different tokenizers produce different token counts for the same text — which affects context window usage, cost, and sometimes model behavior.

In [ ]:
# BPE tokenization with tiktoken (OpenAI's tokenizer)
enc = tiktoken.get_encoding("cl100k_base")  # Used by GPT-4, ChatGPT

text = "The transformer architecture revolutionized natural language processing."
tokens = enc.encode(text)

print("Original text:", text)
print("\nToken IDs:", tokens)
print("\nToken strings:")
for i, tok_id in enumerate(tokens):
    print(f"  {i}: {repr(enc.decode([tok_id]))}")

decoded = enc.decode(tokens)
print("\nDecoded back:", decoded)
print("\nToken count:", len(tokens))

In [ ]:
# Compare token counts across different HuggingFace tokenizers
# Using open models (no HuggingFace auth required)
text = "Generative AI models can produce fluent text, but fluency is not accuracy."

models = ["gpt2", "distilgpt2", "EleutherAI/gpt-neo-125M"]

print("Token counts for the same text across models:\n")
for model_name in models:
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        tokens = tokenizer.encode(text)
        print(f"{model_name}: {len(tokens)} tokens")
    except Exception as e:
        print(f"{model_name}: (skipped - {e})")

# GPT-2 tokenizer for comparison
gpt2_tok = GPT2Tokenizer.from_pretrained("gpt2")
gpt2_tokens = gpt2_tok.encode(text)
print(f"\nGPT-2 token breakdown:")
for i, tid in enumerate(gpt2_tokens):
    print(f"  {i}: {repr(gpt2_tok.decode([tid]))}")

In [ ]:
# How tokenization affects model behavior: token boundaries matter
# Words split across tokens can be harder for models to handle

gpt2_tok = GPT2Tokenizer.from_pretrained("gpt2")

words = ["unhappiness", "antidisestablishmentarianism", "hello", "tokenization"]

print("How many tokens per word? (GPT-2 tokenizer)\n")
for w in words:
    tokens = gpt2_tok.encode(w)
    token_strs = [gpt2_tok.decode([t]) for t in tokens]
    print(f"  '{w}' -> {len(tokens)} tokens: {token_strs}")

---
## 3. Attention Mechanism

### Self-Attention Intuition

Think of self-attention as a **query-key-value** system:

- **Query (Q)**: "What am I looking for?" — each token asks what information it needs
- **Key (K)**: "What do I offer?" — each token describes what it can contribute
- **Value (V)**: "Here's my content" — the actual information passed forward

For each token, we compute **attention scores** (dot product of Q with all Ks), apply softmax to get **attention weights**, then take a weighted sum of the **Values**. Tokens with high attention weight contribute more to the output.

The **scaled** dot-product (dividing by √d_k) prevents gradients from vanishing when the key dimension is large.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled dot-product attention from scratch.
    Q, K, V: (batch, seq_len, d_k)
    """
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, V)
    return output, attn_weights

# Example: 4 tokens, dimension 8
seq_len, d_k = 4, 8
Q = torch.randn(1, seq_len, d_k)
K = torch.randn(1, seq_len, d_k)
V = torch.randn(1, seq_len, d_k)

output, attn_weights = scaled_dot_product_attention(Q, K, V)
print("Attention weights shape:", attn_weights.shape)
print("Output shape:", output.shape)
print("\nAttention weights (each row sums to 1):")
print(attn_weights.squeeze().detach().numpy().round(3))

In [ ]:
# Visualize attention weights as a heatmap
words = ["The", "cat", "sat", "down"]
attn = attn_weights.squeeze().detach().numpy()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(attn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(words)))
ax.set_yticks(range(len(words)))
ax.set_xticklabels(words)
ax.set_yticklabels(words)
ax.set_xlabel("Key (attended to)")
ax.set_ylabel("Query (attending from)")
plt.colorbar(im, ax=ax, label="Attention weight")
plt.title("Self-Attention Weights (random init)")
plt.tight_layout()
plt.show()

---
## 4. The Transformer Architecture

### Encoder-Decoder vs Decoder-Only

- **Encoder-decoder** (e.g., original Transformer, T5, Gemini): The encoder processes the input bidirectionally; the decoder generates output autoregressively, attending to the encoder's output. Best for tasks like translation and summarization where input and output are distinct.

- **Decoder-only** (e.g., GPT, Llama, Claude): A stack of decoder layers only. Each token attends to previous tokens (causal masking). Trained to predict the next token. Simpler, scales well, and has become dominant for general-purpose language modeling.

### Key Components

1. **Multi-head attention**: Run several attention "heads" in parallel, each with its own Q/K/V projections; concatenate and project. Captures different types of relationships.
2. **Feed-forward network**: Two linear layers with a non-linearity (e.g., GELU) in between. Applied per token.
3. **Layer normalization**: Normalize activations before/after sub-layers for training stability.
4. **Positional encoding**: Inject position information (sinusoidal or learned) so the model knows token order.

---
## 5. Text Generation with GPT-2

In [ ]:
model = GPT2LMHeadModel.from_pretrained("gpt2")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

model.eval()
print("GPT-2 loaded. Parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

In [ ]:
def generate_text(prompt, temperature=0.7, max_new_tokens=50, do_sample=True, top_k=50, top_p=1.0):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=do_sample,
            top_k=top_k if do_sample else 0,
            top_p=top_p if do_sample else 1.0,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

prompt = "The future of artificial intelligence"

print("=== Greedy (temperature=0, deterministic) ===")
print(generate_text(prompt, temperature=0.1, do_sample=False))

print("\n=== Sampling (temperature=0.7) ===")
print(generate_text(prompt, temperature=0.7))

print("\n=== High temperature (temperature=1.2, more random) ===")
print(generate_text(prompt, temperature=1.2))

In [ ]:
# Compare decoding strategies: greedy vs top-k vs top-p
prompt = "In machine learning,"

print("Greedy:")
print(generate_text(prompt, temperature=0.01, do_sample=False, max_new_tokens=30))

print("\nTop-k=10:")
print(generate_text(prompt, temperature=0.7, top_k=10, top_p=1.0, max_new_tokens=30))

print("\nTop-p=0.9 (nucleus sampling):")
print(generate_text(prompt, temperature=0.7, top_k=0, top_p=0.9, max_new_tokens=30))

---
## 6. Exercises

### Exercise 1: Tokenize in Multiple Languages

Tokenize the same paragraph in **3 different languages** (e.g., English, Spanish, Chinese) and compare token counts. Use both `tiktoken` and the GPT-2 tokenizer. What do you observe about token efficiency across languages?

In [ ]:
# TODO: Exercise 1 - Tokenize a paragraph in 3 languages and compare token counts
# Example paragraphs (translate to equivalent meaning):
# English: "Machine learning is a subset of artificial intelligence."
# Spanish: "El aprendizaje automático es un subconjunto de la inteligencia artificial."
# Chinese: "机器学习是人工智能的一个子集。"

paragraphs = {
    "English": "Machine learning is a subset of artificial intelligence.",
    "Spanish": "El aprendizaje automático es un subconjunto de la inteligencia artificial.",
    "Chinese": "机器学习是人工智能的一个子集。"
}

# Your code here: use tiktoken and GPT2Tokenizer, print token counts for each
enc = tiktoken.get_encoding("cl100k_base")
gpt2_tok = GPT2Tokenizer.from_pretrained("gpt2")

for lang, text in paragraphs.items():
    tiktoken_count = len(enc.encode(text))
    gpt2_count = len(gpt2_tok.encode(text))
    print(f"{lang}: tiktoken={tiktoken_count}, GPT-2={gpt2_count} tokens")

### Exercise 2: Attention Visualization Enhancement

Modify the attention heatmap to **highlight which words attend most strongly to which**. Add annotations or a different visualization (e.g., arrows, thicker borders) to make the strongest attention links obvious.

In [ ]:
# TODO: Exercise 2 - Enhance attention visualization to highlight strongest links
# Recompute attention (or use attn_weights from Section 3 if already run)
seq_len, d_k = 4, 8
Q = torch.randn(1, seq_len, d_k)
K = torch.randn(1, seq_len, d_k)
V = torch.randn(1, seq_len, d_k)
_, attn_weights = scaled_dot_product_attention(Q, K, V)

words = ["The", "cat", "sat", "down"]
attn = attn_weights.squeeze().detach().numpy()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(attn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(words)))
ax.set_yticks(range(len(words)))
ax.set_xticklabels(words)
ax.set_yticklabels(words)

# Your enhancement: add text annotations for values, or circle max per row
for i in range(len(words)):
    for j in range(len(words)):
        val = attn[i, j]
        color = "white" if val > 0.5 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", color=color, fontsize=9)

ax.set_xlabel("Key (attended to)")
ax.set_ylabel("Query (attending from)")
plt.colorbar(im, ax=ax, label="Attention weight")
plt.title("Self-Attention with Value Annotations")
plt.tight_layout()
plt.show()

### Exercise 3: Text Generation Analysis

Generate text with GPT-2 using **different prompts and temperatures**. Try: (a) a factual prompt, (b) a creative prompt, (c) a technical prompt. For each, compare low vs high temperature. Write 2-3 sentences analyzing the quality and coherence of the outputs.

In [ ]:
# TODO: Exercise 3 - Generate with different prompts and temperatures, analyze quality
prompts = [
    "The capital of France is",
    "Once upon a time, in a distant galaxy,",
    "The gradient descent algorithm"
]

for prompt in prompts:
    print(f"Prompt: '{prompt}'")
    print("  Low temp (0.2):", generate_text(prompt, temperature=0.2, max_new_tokens=25)[:100] + "...")
    print("  High temp (1.0):", generate_text(prompt, temperature=1.0, max_new_tokens=25)[:100] + "...")
    print()

# Your analysis: Which temperature worked best for each prompt type? Why?

---
## 7. Responsible AI Reflection

LLMs can generate fluent, confident text on virtually any topic. But **fluency is not accuracy**, and **confidence is not correctness**. As you begin this course, consider:

> **What is the most dangerous capability of LLMs — not technically, but societally?** Is it the ability to generate misinformation at scale, the appearance of expertise without understanding, or something else entirely?

**Write 2-3 sentences defending your position.**

In [ ]:
# Your reflection (2-3 sentences):
# 
# 
# 

---
## 8. Summary & Next Steps

### What We Covered

- **Generative AI landscape**: From RNNs to transformers to today's frontier models (GPT, Claude, Gemini, Llama, Mistral)
- **Tokenization**: BPE with tiktoken and HuggingFace; how token boundaries affect model behavior
- **Attention**: Query-key-value intuition; scaled dot-product attention from scratch; visualization
- **Transformer architecture**: Encoder-decoder vs decoder-only; multi-head attention, feed-forward, layer norm, positional encoding
- **Text generation**: GPT-2 with greedy, sampling, top-k, and top-p decoding

### Next Steps

- **Week 2**: AI for Academic Research — using deep research tools with verification workflows
- **Week 3**: Prompt Engineering — zero-shot, few-shot, chain-of-thought

---

*Notebook complete. CC BY 4.0 —  | Applied Generative AI*